# Example 3: Streaming vs Non-streaming —— 性能对比

对同一个请求分别使用 streaming 和 non-streaming 模式，对比：
- **TTFT（首 token 延迟）**
- **端到端总耗时**
- **输出内容一致性**

In [ ]:
import time
from openai import OpenAI

API_KEY = "sk-你的APIKey"  # TODO: 替换

client = OpenAI(
    base_url="https://tokenhub.tencentmaas.com/v1",
    api_key=API_KEY,
)

prompt = """请用 Python 实现一个 LRU (Least Recently Used) 缓存，
包含 get 和 put 方法，并解释时间复杂度。"""

## 1. Non-streaming 调用

In [ ]:
print("-" * 40)
print("Non-streaming (stream=False)")
print("-" * 40)

start = time.time()
response = client.chat.completions.create(
    model="hy3",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7, top_p=1.0, stream=False,
    extra_body={"reasoning_effort": "no_think"},
)
non_stream_end = time.time()

non_stream_content = response.choices[0].message.content
usage = response.usage

print(f"总耗时:    {non_stream_end - start:.2f}s")
print(f"TTFT:      不可测量（等待全部完成后才返回）")
print(f"prompt tokens:  {usage.prompt_tokens}")
print(f"completion tokens: {usage.completion_tokens}")
print(f"总 tokens: {usage.total_tokens}")

## 2. Streaming 调用

In [ ]:
print("-" * 40)
print("Streaming (stream=True)")
print("-" * 40)

start = time.time()
first_token_time = None

response = client.chat.completions.create(
    model="hy3",
    messages=[{"role": "user", "content": prompt}],
    temperature=0.7, top_p=1.0, stream=True,
    extra_body={"reasoning_effort": "no_think"},
)

stream_full_content = ""
for chunk in response:
    if first_token_time is None:
        first_token_time = time.time()
    if chunk.choices[0].delta.content:
        stream_full_content += chunk.choices[0].delta.content
    if chunk.choices[0].finish_reason:
        stream_end = time.time()

print(f"TTFT:      {first_token_time - start:.2f}s")
print(f"总耗时:    {stream_end - start:.2f}s")
print(f"输出长度:  {len(stream_full_content)} 字符")

## 3. 对比总结

In [ ]:
print("=" * 50)
print("对比总结")
print("=" * 50)

print(f"""
指标                Non-streaming    Streaming
----------------------------------------------
TTFT (首 token)     N/A              {first_token_time - start:.2f}s
总耗时              {non_stream_end - non_stream_end:.2f}s            {stream_end - start:.2f}s
输出长度            {len(non_stream_content)} 字符        {len(stream_full_content)} 字符
""")

match = non_stream_content == stream_full_content
print(f"输出内容一致: {'✓' if match else '⚠ 略有差异（采样随机性）'}")
print(f"\n结论: streaming TTFT 仅 {(first_token_time - start):.2f}s，用户感知响应极快")